User_details Silver

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.user_details_silver AS
SELECT
    UPPER(TRIM(username)) AS username,
    diplayname,
    department,
    mail,
    samdomain_name,
    lab_name,
    region,
    lab_source
FROM workspace.bronze.user_details_bronze
WHERE username IS NOT NULL
  AND lab_name IS NOT NULL;

In [0]:
%sql
select * from workspace.silver.user_details_silver
where username = 'KITISAKP'

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.user_details_bronze

Audit_log Silver

In [0]:
# %sql
# CREATE OR REPLACE TABLE workspace.silver.audit_defects_silver AS
# SELECT
#     UPPER(TRIM(user)) AS user,
#     data_provider,
#     REPLACE(data_provider, 'RNBI_MANUFAC_', '') AS application_name,
#     event_type,
#     start_datetime,
#     event_duration_seconds,
#     object,
#     top_level_folder,
#     month,
#     day,
#     year,
#     week_day,
#     hour
# FROM workspace.bronze.audit_bronze
# WHERE user IS NOT NULL
#   AND data_provider IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.audit_defects_silver AS
SELECT
    UPPER(TRIM(user)) AS user,
    data_provider,
    REPLACE(data_provider, 'RNBI_MANUFAC_', '') AS application_name,
    event_type,
    DATE_FORMAT(
      COALESCE(
        TRY_TO_TIMESTAMP(REGEXP_REPLACE(CAST(start_datetime AS STRING), '\\.\\d+$', ''), 'yyyy-MM-dd HH:mm:ss'),
        TRY_TO_TIMESTAMP(REGEXP_REPLACE(CAST(start_datetime AS STRING), '\\.\\d+$', ''), 'yyyy/MM/dd HH:mm:ss')
      ),
      'yyyy/MM/dd HH:mm:ss'
    ) AS start_datetime,
    event_duration_seconds,
    object,
    top_level_folder,
    month,
    day,
    year,
    week_day,
    hour
FROM workspace.bronze.audit_bronze
WHERE user IS NOT NULL
  AND data_provider IS NOT NULL;

In [0]:
%sql
SELECT application_name, COUNT(*)
FROM workspace.silver.audit_defects_silver
GROUP BY application_name
ORDER BY COUNT(*) DESC;

In [0]:
%sql
select * from workspace.silver.audit_defects_silver

LADP Silver

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.ldap_users_silver AS
SELECT
    upper(trim(username)) AS username,
    diplayname,
    department,
    mail,
    samdomainname AS samdomain_name,
    CAST(NULL AS STRING) AS lab_name,
    CAST(NULL AS STRING) AS region,
    CAST(NULL AS STRING) AS lab_source
FROM workspace.bronze.ldap_users_bronze
WHERE username IS NOT NULL;

In [0]:
%sql
SELECT * FROM workspace.silver.ldap_users_silver
LIMIT 10;

In [0]:
# %sql
# -- Check how many users have multiple lab assignments
# SELECT 
#     'user_details' AS source,
#     COUNT(*) AS total_rows,
#     COUNT(DISTINCT username) AS distinct_users,
#     COUNT(*) - COUNT(DISTINCT username) AS duplicate_users
# FROM workspace.silver.user_details_silver

# UNION ALL

# SELECT 
#     'ldap_users' AS source,
#     COUNT(*) AS total_rows,
#     COUNT(DISTINCT username) AS distinct_users,
#     COUNT(*) - COUNT(DISTINCT username) AS duplicate_users
# FROM workspace.silver.ldap_users_silver;